In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "research":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "config" / "config.yaml").exists():
    PROJECT_ROOT = Path(r"C:\Users\vishn\Desktop\NLP\End-to-end-Text-Summarizer")

os.chdir(PROJECT_ROOT)
print(f"Project root: {Path.cwd()}")
print(f"Config exists: {(Path('config') / 'config.yaml').exists()}")
print(f"Params exists: {Path('params.yaml').exists()}")


In [ ]:
# Dependencies are managed by requirements.txt for this local project.
# Run this in a terminal if needed:
# pip install -r requirements.txt


In [ ]:
import transformers
import datasets
import torch

print(f"transformers: {transformers.__version__}")
print(f"datasets: {datasets.__version__}")
print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")


In [ ]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from tqdm import tqdm
import torch


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
model_ckpt = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

In [ ]:
from pathlib import Path

from textSummarizer.pipeline.stage_01_data_ingestion import DataIngestionTrainingPipeline

DATASET_PATH = Path("artifacts/data_ingestion/samsum_dataset")
if not DATASET_PATH.exists():
    DataIngestionTrainingPipeline().main()

print(f"Dataset path: {DATASET_PATH}")


In [ ]:
dataset_samsum = load_from_disk(str(DATASET_PATH))
dataset_samsum


In [ ]:
split_lengths = [len(dataset_samsum[split])for split in dataset_samsum]

print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")
print("\nDialogue:")

print(dataset_samsum["test"][1]["dialogue"])

print("\nSummary:")

print(dataset_samsum["test"][1]["summary"])

In [ ]:
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(
        example_batch['dialogue'],
        max_length=1024,
        truncation=True
    )
    try:
        target_encodings = tokenizer(
            text_target=example_batch['summary'],
            max_length=128,
            truncation=True
        )
    except TypeError:
        with tokenizer.as_target_tokenizer():
            target_encodings = tokenizer(
                example_batch['summary'],
                max_length=128,
                truncation=True
            )

    return {
        'input_ids': input_encodings['input_ids'],
        'attention_mask': input_encodings['attention_mask'],
        'labels': target_encodings['input_ids']
    }


In [ ]:

dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features, batched = True)


In [ ]:
dataset_samsum_pt["train"]

In [ ]:
#training

from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [ ]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir='artifacts/research/pegasus-samsum',
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=500,
    save_steps=1e6,
    gradient_accumulation_steps=16,
    fp16=torch.cuda.is_available(),
    report_to='none'
)


In [ ]:
trainer = Trainer(model=model_pegasus, args=trainer_args,
                  data_collator=seq2seq_data_collator,
                  train_dataset=dataset_samsum_pt["train"],
                   eval_dataset=dataset_samsum_pt['validation'])

In [ ]:
trainer.train()

In [ ]:
from rouge_score import rouge_scorer


In [ ]:
def generate_batch_sized_chunks(list_of_elements, batch_size):
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]


def calculate_metric_on_test_ds(dataset, model, tokenizer,
                                batch_size=16, device=device,
                                column_text="article",
                                column_summary="highlights"):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL", "rougeLsum"],
        use_stemmer=True,
    )
    totals = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0, "rougeLsum": 0.0}
    count = 0

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)):

        inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                           padding="max_length", return_tensors="pt")
        summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                                   attention_mask=inputs["attention_mask"].to(device),
                                   length_penalty=0.8, num_beams=8, max_length=128)
        decoded_summaries = tokenizer.batch_decode(
            summaries,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        for prediction, reference in zip(decoded_summaries, target_batch):
            scores = scorer.score(reference, prediction)
            for rouge_name in totals:
                totals[rouge_name] += scores[rouge_name].fmeasure
            count += 1

    return {rouge_name: score / max(count, 1) for rouge_name, score in totals.items()}


In [ ]:
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]


In [ ]:
score = calculate_metric_on_test_ds(
    dataset_samsum['test'], trainer.model, tokenizer,
    batch_size=2,
    column_text='dialogue',
    column_summary='summary'
)

rouge_dict = dict((rn, score[rn]) for rn in rouge_names)

pd.DataFrame(rouge_dict, index=[f'pegasus'])


In [ ]:
model_pegasus.save_pretrained('pegasus-samsum-model')

In [ ]:
tokenizer.save_pretrained('tokenizer')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('tokenizer')


In [ ]:
gen_kwargs = {"length_penalty": 0.8, "num_beams":8, "max_length": 128}

sample_text = dataset_samsum["test"][0]['dialogue']

reference = dataset_samsum["test"][0]['summary']

# The pipeline function is causing an 'Unknown task summarization' error.
# We will perform summarization manually using the model and tokenizer directly.
# pipe = pipeline("summarization", model="pegasus-samsum-model",tokenizer=tokenizer)

print('Dialogue: ')
print(sample_text)

print("\nReference Summary: ")
print(reference)

# Manually generate summary
inputs = tokenizer(sample_text, max_length=1024, truncation=True, padding="max_length", return_tensors="pt")
generated_summary_ids = model_pegasus.generate(input_ids=inputs["input_ids"].to(device),
                                             attention_mask=inputs["attention_mask"].to(device),
                                             **gen_kwargs)
model_summary = tokenizer.decode(generated_summary_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)

print("\nModel Summary: ")
print(model_summary)